In [2]:
import shap
from xgboost.sklearn import XGBClassifier
from sklearn.metrics import r2_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [3]:
def create_surrogate(X, predictions, sur_parameters):
    # Create surrogate xgboost model
    surrogate = XGBClassifier(**sur_parameters)
    surrogate.fit(X, predictions)
    sur_predictions = surrogate.predict(X)
    external_fidelity = r2_score(predictions, sur_predictions)

    return surrogate, external_fidelity

def calc_shap(model, x_explain, approximate=False):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(x_explain, approximate=approximate)
    return shap_values, explainer

In [4]:
from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_breast_cancer(return_X_y=True)
X_train, _, y_train, _ = train_test_split(X, y, random_state=42)
original_model = MLPClassifier()
original_model.fit(X_train, y_train)
predictions = original_model.predict(X)

# Use agnostic explainer
sur_parameters = {'n_estimators': 100, 'max_depth': 8}
surrogate_model, external_fidelity = create_surrogate(X,
                                                      predictions,
                                                      sur_parameters)
# get the shapley values for the whole dataset
shap_values, explainer = calc_shap(surrogate_model, X)

The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].


[13:10:01] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.0/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


ntree_limit is deprecated, use `iteration_range` or model slicing instead.


## Manual predictions

In [39]:
local = X[24]
from numpy import asarray
input = asarray([local])

In [46]:
# ----------- Extract local predictions through the loops -----------
def grab_predictions(instance,iteration):
    prediction = surrogate_model.predict(instance)
    preds = []
    if prediction.item(0) == 1:
        preds.append("Iteration Number %d: " %iteration +"Breast cancer detected :(")
    else:
        preds.append("Iteration Number %d: " %iteration +"You are okay :)")
    preds = pd.DataFrame(preds)  
    preds.to_csv(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\L_E_MASHAP\predictions.txt', header=None, index=None, sep=' ', mode='a')

# This will show us how the prediction changes for the instance. Will be interesting to compare with 
# the input perturbation script to see if it also affects predictions alongside explanations

In [47]:
for i in range(10):
    local = X[i]
    from numpy import asarray
    input = asarray([local])
    grab_predictions(input,i)

## Explain the prediction

In [44]:
shap_values, explainer = calc_shap(surrogate_model, input)

ntree_limit is deprecated, use `iteration_range` or model slicing instead.


In [25]:
import pandas as pd
import numpy as np

iteration_number = 1

# ----------- Extract global explanations -----------
vals= np.abs(shap_values).mean(0)

X_train = pd.DataFrame(X_train)

feature_importance = pd.DataFrame(list(zip(X_train.columns,vals)),columns=['col_name','feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'],ascending=False,inplace=True)
feature_importance["col_name"] =  'Feature ' + feature_importance["col_name"].astype(str)

feature_importance = feature_importance.head()

feature_importance.to_csv(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\L_E_MASHAP\features.txt', header=None, index=None, sep=' ', mode='a')

with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\L_E_MASHAP\features.txt','a') as file:
    file.write("\nRun number: %d\n\n" %iteration_number)
file.close()